# 🛡️ LoanShield — Lending Fraud & Default Detection Engine
### Ensemble Methods: Decision Tree · Random Forest (Bagging) · XGBoost (Boosting)

**Domain:** FinTech / Lending Risk  
**Dataset:** HMEQ — Home Equity Loan Default (5,960 rows, 13 features)  
**Target:** `BAD` — 1 = defaulted / severely delinquent, 0 = repaid  

---
**Three-Act Narrative:**
- 🔴 **Act 1 — Fraud Landscape:** Who are the defaulters?
- ⚔️ **Act 2 — Ensemble Showdown:** Which model wins and why?
- 💰 **Act 3 — Business Verdict:** What does this save the lender?


## Cell 1 — Imports & Global Settings

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection      import train_test_split, cross_val_score
from sklearn.preprocessing        import LabelEncoder
from sklearn.tree                 import DecisionTreeClassifier
from sklearn.ensemble             import RandomForestClassifier
from sklearn.metrics              import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)
from xgboost import XGBClassifier

# ── Dark-theme plot settings (consistent with portfolio) ─────────────────────
plt.rcParams.update({
    "figure.facecolor": "#0f0f0f",
    "axes.facecolor":   "#1a1a2e",
    "axes.edgecolor":   "#444",
    "axes.labelcolor":  "#e0e0e0",
    "xtick.color":      "#aaa",
    "ytick.color":      "#aaa",
    "text.color":       "#e0e0e0",
    "grid.color":       "#2a2a4a",
    "figure.dpi":       150,
    "legend.facecolor": "#1a1a2e",
    "legend.edgecolor": "#444",
})

# ── Colour palette (used consistently across all charts) ─────────────────────
COLOR_FRAUD   = "#ef5350"   # red   — fraudulent / high-risk
COLOR_LEGIT   = "#66bb6a"   # green — legitimate / low-risk
COLOR_DT      = "#78909c"   # gray  — baseline Decision Tree
COLOR_RF      = "#42a5f5"   # blue  — Random Forest
COLOR_XGB     = "#ffca28"   # amber — XGBoost
COLOR_NEUTRAL = "#7e57c2"   # purple — neutral highlights

print("✓ Libraries loaded")
print(f"  XGBoost version : {__import__('xgboost').__version__}")


## Cell 2 — Load Dataset

**How to get the data:**
1. Go to: https://www.kaggle.com/datasets/ajay1735/hmeq-data
2. Download `hmeq.csv` (~350 KB)
3. In Colab: **Files → Upload → select hmeq.csv**

**Column legend:**

| Column | Description |
|--------|-------------|
| `BAD` | **TARGET** — 1 = defaulted/severely delinquent, 0 = repaid |
| `LOAN` | Amount of the loan request ($) |
| `MORTDUE` | Amount due on existing mortgage ($) |
| `VALUE` | Current value of the property ($) |
| `REASON` | Reason for loan (HomeImprovement / DebtConsolidation) |
| `JOB` | Occupational categories (6 types) |
| `YOJ` | Years at present job |
| `DEROG` | Number of major derogatory reports (serious red flag) |
| `DELINQ` | Number of delinquent credit lines |
| `CLAGE` | Age of oldest credit line (months) |
| `NINQ` | Number of recent credit inquiries |
| `CLNO` | Number of credit lines |
| `DEBTINC` | Debt-to-income ratio (%) — most predictive feature |


In [ ]:
# ── Upload hmeq.csv to Colab first, then run this cell ───────────────────────
raw_df = pd.read_csv("hmeq.csv")

print(f"  Shape   : {raw_df.shape}")
print(f"  Columns : {list(raw_df.columns)}")
print(f"\n  First 3 rows:")
print(raw_df.head(3).to_string())
print(f"\n  Target distribution (BAD):")
print(raw_df["BAD"].value_counts())
print(f"\n  Fraud rate: {raw_df['BAD'].mean()*100:.1f}%")


## Cell 3 — Act 1: Fraud Landscape — EDA & Missing Value Audit

> Before building any model, a lender needs to understand the **risk landscape**.
> Three questions:
> 1. **How rare is fraud?** — defines the class imbalance strategy
> 2. **What do fraudsters look like** vs legitimate borrowers?
> 3. **Which features are dirty** and need cleaning?


In [ ]:
df = raw_df.copy()

# 3.1  Basic audit
print("Shape:", df.shape)
print("\nMissing values per column:")
missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
audit = pd.DataFrame({"count": missing, "pct": missing_pct})
print(audit[audit["count"] > 0].to_string())

# 3.2  Class distribution
fraud_count = df["BAD"].sum()
legit_count = len(df) - fraud_count
fraud_rate  = fraud_count / len(df) * 100
print(f"\nClass distribution:")
print(f"  Legitimate  (BAD=0)    : {legit_count:,}  ({100-fraud_rate:.1f}%)")
print(f"  Default/Fraud (BAD=1)  : {fraud_count:,}  ({fraud_rate:.1f}%)")
print(f"  Imbalance ratio        : {legit_count/fraud_count:.1f}:1")

# 3.3  Numeric feature means split by class
print("\nKey feature means — fraud vs legitimate:")
num_feats = ["LOAN", "MORTDUE", "VALUE", "DEROG", "DELINQ", "DEBTINC", "YOJ", "CLAGE"]
summary   = df.groupby("BAD")[num_feats].mean().round(2)
summary.index = ["Legitimate", "Fraud/Default"]
print(summary.to_string())


## Cell 4 — Act 1: EDA Visualisation (Fraud Landscape Dashboard)

In [ ]:
# ── Chart 1 of 4: Class Distribution ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.bar(["Legitimate\n(BAD=0)", "Default / Fraud\n(BAD=1)"],
              [legit_count, fraud_count],
              color=[COLOR_LEGIT, COLOR_FRAUD], width=0.5, edgecolor="#333")
ax.set_title("Class Distribution — How Rare Is Fraud?", color="#e0e0e0", fontsize=13)
ax.set_ylabel("Number of Applications")
for bar, count in zip(bars, [legit_count, fraud_count]):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 40,
            f"{count:,}\n({count/len(df)*100:.1f}%)",
            ha="center", va="bottom", color="#e0e0e0", fontsize=11, fontweight="bold")
ax.set_ylim(0, legit_count * 1.2)
ax.text(0.98, 0.95,
        f"Imbalance ratio\n{legit_count/fraud_count:.1f} : 1",
        transform=ax.transAxes, ha="right", va="top",
        color="#ffca28", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#2a2a4a", edgecolor="#ffca28"))
plt.tight_layout()
plt.savefig("act1_class_distribution.png", bbox_inches="tight", dpi=150)
plt.show()
print("[SAVED] act1_class_distribution.png")

# ── Chart 2 of 4: Debt-to-Income Distribution ─────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
legit_di = df[df["BAD"] == 0]["DEBTINC"].dropna()
fraud_di = df[df["BAD"] == 1]["DEBTINC"].dropna()
ax.hist(legit_di, bins=50, alpha=0.65, color=COLOR_LEGIT,
        label=f"Legitimate (n={len(legit_di):,})", density=True)
ax.hist(fraud_di, bins=50, alpha=0.65, color=COLOR_FRAUD,
        label=f"Default/Fraud (n={len(fraud_di):,})", density=True)
ax.axvline(legit_di.mean(), color=COLOR_LEGIT, lw=2, ls="--",
           label=f"Legit mean: {legit_di.mean():.1f}%")
ax.axvline(fraud_di.mean(), color=COLOR_FRAUD, lw=2, ls="--",
           label=f"Fraud mean: {fraud_di.mean():.1f}%")
ax.set_title("Debt-to-Income Ratio — Key Risk Signal", color="#e0e0e0", fontsize=13)
ax.set_xlabel("Debt-to-Income (%)")
ax.set_ylabel("Density")
ax.legend(fontsize=9, framealpha=0.4)
plt.tight_layout()
plt.savefig("act1_debt_to_income.png", bbox_inches="tight", dpi=150)
plt.show()
print("[SAVED] act1_debt_to_income.png")

# ── Chart 3 of 4: Derogatory Reports Heatmap ──────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
derog_by_class = df.groupby("BAD")["DEROG"].value_counts(normalize=True).unstack(fill_value=0)
derog_by_class = derog_by_class[[c for c in derog_by_class.columns if c <= 5]]
derog_by_class.index = ["Legitimate", "Fraud/Default"]
im = ax.imshow(derog_by_class.values, aspect="auto", cmap="RdYlGn_r",
               vmin=0, vmax=derog_by_class.values.max())
ax.set_xticks(range(len(derog_by_class.columns)))
ax.set_xticklabels([f"{c} report{'s' if c != 1 else ''}" for c in derog_by_class.columns], fontsize=10)
ax.set_yticks([0, 1])
ax.set_yticklabels(["Legitimate", "Fraud/Default"], fontsize=11)
ax.set_title("Derogatory Reports Distribution by Class", color="#e0e0e0", fontsize=13)
for i in range(2):
    for j in range(len(derog_by_class.columns)):
        ax.text(j, i, f"{derog_by_class.values[i, j]:.0%}",
                ha="center", va="center", color="white", fontsize=10, fontweight="bold")
plt.colorbar(im, ax=ax, label="Proportion")
plt.tight_layout()
plt.savefig("act1_derogatory_reports.png", bbox_inches="tight", dpi=150)
plt.show()
print("[SAVED] act1_derogatory_reports.png")

# ── Chart 4 of 4: Missing Data Heatmap ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
missing_by_class = df.groupby("BAD").apply(lambda x: x.isnull().mean() * 100)
missing_by_class.index = ["Legitimate", "Fraud/Default"]
missing_by_class = missing_by_class.drop(columns=["BAD"])
missing_by_class = missing_by_class.loc[:, missing_by_class.max() > 0]
im2 = ax.imshow(missing_by_class.values, aspect="auto", cmap="YlOrRd",
                vmin=0, vmax=missing_by_class.values.max())
ax.set_xticks(range(len(missing_by_class.columns)))
ax.set_xticklabels(missing_by_class.columns, rotation=35, ha="right", fontsize=10)
ax.set_yticks([0, 1])
ax.set_yticklabels(["Legitimate", "Fraud/Default"], fontsize=11)
ax.set_title("Missing Data % by Class — Imputation Needed", color="#e0e0e0", fontsize=13)
for i in range(2):
    for j in range(len(missing_by_class.columns)):
        ax.text(j, i, f"{missing_by_class.values[i, j]:.1f}%",
                ha="center", va="center", color="#111", fontsize=9, fontweight="bold")
plt.colorbar(im2, ax=ax, label="% Missing")
plt.tight_layout()
plt.savefig("act1_missing_data.png", bbox_inches="tight", dpi=150)
plt.show()
print("[SAVED] act1_missing_data.png")


## Cell 5 — Data Cleaning & Preprocessing

**Imputation strategy:**

| Column type | Strategy | Reason |
|-------------|----------|--------|
| Categorical (`REASON`, `JOB`) | Fill with **mode** | Low-cardinality; no meaningful "unknown" class |
| Numeric (`DEBTINC`, `DEROG`, etc.) | **Class-conditional median** | A fraudster's missing DEBTINC has a different distribution than a legit borrower's — preserves that signal |


In [ ]:
df_clean = df.copy()

# 5.1  Categorical columns — fill with mode
for col in ["REASON", "JOB"]:
    mode_val = df_clean[col].mode()[0]
    df_clean[col].fillna(mode_val, inplace=True)
    print(f"[5.1]  {col:8s} — filled {df[col].isnull().sum()} NaNs with mode='{mode_val}'")

# 5.2  Numeric columns — class-conditional median imputation
num_cols_with_na = [c for c in df_clean.select_dtypes(include=np.number).columns
                    if c != "BAD" and df_clean[c].isnull().sum() > 0]

for col in num_cols_with_na:
    for cls in [0, 1]:
        median_val = df_clean.loc[df_clean["BAD"] == cls, col].median()
        mask       = (df_clean["BAD"] == cls) & (df_clean[col].isnull())
        df_clean.loc[mask, col] = median_val
    n_filled = df[col].isnull().sum()
    print(f"[5.2]  {col:8s} — class-conditional median imputation ({n_filled} NaNs filled)")

# 5.3  Verify zero NaNs
assert df_clean.isnull().sum().sum() == 0, "FATAL: residual NaNs after cleaning"
print(f"\n[5.3]  ✓ Zero NaNs confirmed — shape: {df_clean.shape}")

# 5.4  Encode categorical features
le_reason = LabelEncoder()
le_job    = LabelEncoder()
df_clean["REASON_ENC"] = le_reason.fit_transform(df_clean["REASON"])
df_clean["JOB_ENC"]    = le_job.fit_transform(df_clean["JOB"])

print(f"\n[5.4]  REASON categories : {list(le_reason.classes_)}")
print(f"       JOB    categories : {list(le_job.classes_)}")

# 5.5  Feature matrix & target vector
FEATURE_COLS = ["LOAN", "MORTDUE", "VALUE", "YOJ", "DEROG", "DELINQ",
                "CLAGE", "NINQ", "CLNO", "DEBTINC", "REASON_ENC", "JOB_ENC"]

X = df_clean[FEATURE_COLS]
y = df_clean["BAD"]

print(f"\n[5.5]  Feature matrix shape : {X.shape}")
print(f"       Target vector shape  : {y.shape}")
print(f"       Feature names        : {FEATURE_COLS}")


## Cell 6 — Train / Test Split (Stratified)

> With ~20% fraud prevalence, a random split risks concentrating fraudsters in one half.
> `stratify=y` preserves the exact fraud rate in **both** train and test splits.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,
    random_state = 42,
    stratify     = y          # preserves fraud rate in both splits
)

print(f"  Train size : {X_train.shape[0]:,} rows  (fraud rate: {y_train.mean()*100:.1f}%)")
print(f"  Test  size : {X_test.shape[0]:,}  rows  (fraud rate: {y_test.mean()*100:.1f}%)")
print(f"\n  ✓ Stratification confirmed — fraud rate preserved in both splits")


## Cell 7 — Act 2: Model 1 of 3 — Decision Tree (Baseline)

**Purpose:** The Decision Tree is our strawman. It deliberately shows *why a single tree is not enough*:
- Overfits the training set (high variance)
- No mechanism to reduce error across diverse samples
- Sets up the narrative: **"here's the problem ensembles solve"**

> `class_weight="balanced"` — penalises missing a fraud proportionally to the imbalance ratio.
> Without it, the model predicts everyone as legitimate (80% accuracy, 0% fraud recall — useless).


In [ ]:
dt = DecisionTreeClassifier(
    max_depth    = 6,             # limit depth to prevent extreme overfitting
    class_weight = "balanced",    # compensate for 80:20 imbalance
    random_state = 42
)
dt.fit(X_train, y_train)

y_pred_dt  = dt.predict(X_test)
y_proba_dt = dt.predict_proba(X_test)[:, 1]
auc_dt     = roc_auc_score(y_test, y_proba_dt)
ap_dt      = average_precision_score(y_test, y_proba_dt)

print(f"  ROC-AUC          : {auc_dt:.4f}")
print(f"  Average Precision: {ap_dt:.4f}")
print(f"\n  Classification Report:")
print(classification_report(y_test, y_pred_dt, target_names=["Legit", "Fraud/Default"]))

# Cross-validation exposes the variance problem
cv_scores_dt = cross_val_score(dt, X_train, y_train, cv=5, scoring="roc_auc")
print(f"  5-Fold CV ROC-AUC: {cv_scores_dt.mean():.4f} ± {cv_scores_dt.std():.4f}")
print(f"  (High std = high variance — this is the problem ensembles fix)")


## Cell 8 — Act 2: Model 2 of 3 — Random Forest (Bagging)

**What is Bagging?**
1. Draw N random samples **with replacement** from training data (bootstrap)
2. Train a full Decision Tree on each bootstrap sample
3. At each split, only consider a **random subset of features** (`max_features="sqrt"`)
4. Aggregate predictions by **majority vote**

**Why it beats a single tree:**  
Each tree sees slightly different data + different features → errors are **uncorrelated** →  
averaging cancels individual variance while compounding shared signal.


In [ ]:
rf = RandomForestClassifier(
    n_estimators = 200,
    max_features = "sqrt",        # √n_features per split — enforces tree diversity
    max_depth    = 12,            # let trees grow deep; bagging handles variance
    class_weight = "balanced",
    n_jobs       = -1,            # use all CPU cores
    random_state = 42
)
rf.fit(X_train, y_train)

y_pred_rf  = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]
auc_rf     = roc_auc_score(y_test, y_proba_rf)
ap_rf      = average_precision_score(y_test, y_proba_rf)

print(f"  ROC-AUC          : {auc_rf:.4f}  (DT was {auc_dt:.4f} — delta: +{auc_rf-auc_dt:.4f})")
print(f"  Average Precision: {ap_rf:.4f}  (DT was {ap_dt:.4f} — delta: +{ap_rf-ap_dt:.4f})")
print(f"\n  Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=["Legit", "Fraud/Default"]))

cv_scores_rf = cross_val_score(rf, X_train, y_train, cv=5, scoring="roc_auc")
print(f"  5-Fold CV ROC-AUC: {cv_scores_rf.mean():.4f} ± {cv_scores_rf.std():.4f}")
print(f"  (Lower std than DT = lower variance — bagging working as intended)")


## Cell 9 — Act 2: Model 3 of 3 — XGBoost (Gradient Boosting)

**What is Boosting?**  
Where bagging trains trees **in parallel** on independent samples, boosting trains trees **sequentially** —  
each new tree focuses on correcting the mistakes of the previous ensemble.

**Gradient Boosting specifically:**
1. Start with a constant prediction
2. Compute residuals (errors)
3. Fit a tree to predict those residuals (the "gradient")
4. Add `tree × learning_rate` to ensemble
5. Repeat — each tree patches errors left by all previous trees

> `scale_pos_weight = count(BAD=0) / count(BAD=1)` — tells XGBoost missing a fraud is ~4x costlier than a false alarm.


In [ ]:
# Calculate imbalance weight for XGBoost
scale_pos_wt = (y_train == 0).sum() / (y_train == 1).sum()
print(f"  scale_pos_weight = {scale_pos_wt:.2f}  (fraud upweighted by {scale_pos_wt:.1f}x)")

xgb = XGBClassifier(
    n_estimators      = 300,
    learning_rate     = 0.05,     # small steps — each tree corrects a little
    max_depth         = 5,        # shallow trees = weak learners (boosting prefers this)
    subsample         = 0.8,      # row subsampling — reduces overfitting
    colsample_bytree  = 0.8,      # column subsampling per tree
    scale_pos_weight  = scale_pos_wt,   # imbalance correction
    eval_metric       = "auc",
    use_label_encoder = False,
    random_state      = 42,
    n_jobs            = -1
)
xgb.fit(X_train, y_train, verbose=False)

y_pred_xgb  = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]
auc_xgb     = roc_auc_score(y_test, y_proba_xgb)
ap_xgb      = average_precision_score(y_test, y_proba_xgb)

print(f"  ROC-AUC          : {auc_xgb:.4f}  (RF was {auc_rf:.4f} — delta: {auc_xgb-auc_rf:+.4f})")
print(f"  Average Precision: {ap_xgb:.4f}  (RF was {ap_rf:.4f} — delta: {ap_xgb-ap_rf:+.4f})")
print(f"\n  Classification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=["Legit", "Fraud/Default"]))

cv_scores_xgb = cross_val_score(xgb, X_train, y_train, cv=5, scoring="roc_auc")
print(f"  5-Fold CV ROC-AUC: {cv_scores_xgb.mean():.4f} ± {cv_scores_xgb.std():.4f}")


## Cell 10 — Act 2: Model Comparison Dashboard

We evaluate using **both** ROC-AUC and Precision-Recall AUC:
- **ROC-AUC** can look optimistic on imbalanced data
- **PR-AUC** focuses purely on the fraud class — harder to game, more meaningful for a lender


In [ ]:
# ── Chart 5 of 12: ROC Curves ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
for name, proba, color, auc_val in [
    ("Decision Tree",   y_proba_dt,  COLOR_DT,  auc_dt),
    ("Random Forest",   y_proba_rf,  COLOR_RF,  auc_rf),
    ("XGBoost",         y_proba_xgb, COLOR_XGB, auc_xgb),
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{name}  AUC={auc_val:.4f}")
ax.plot([0, 1], [0, 1], color="#555", lw=1, ls="--", label="Random baseline")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Decision Tree vs Random Forest vs XGBoost", color="#e0e0e0", fontsize=13)
ax.legend(fontsize=10, framealpha=0.4)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig("act2_roc_curves.png", bbox_inches="tight", dpi=150)
plt.show()
print("[SAVED] act2_roc_curves.png")

# ── Chart 6 of 12: Precision-Recall Curves ────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
for name, proba, color, ap_val in [
    ("Decision Tree",   y_proba_dt,  COLOR_DT,  ap_dt),
    ("Random Forest",   y_proba_rf,  COLOR_RF,  ap_rf),
    ("XGBoost",         y_proba_xgb, COLOR_XGB, ap_xgb),
]:
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ax.plot(rec, prec, color=color, lw=2, label=f"{name}  AP={ap_val:.4f}")
baseline_pr = y_test.mean()
ax.axhline(baseline_pr, color="#555", lw=1, ls="--",
           label=f"Random baseline (AP={baseline_pr:.2f})")
ax.set_xlabel("Recall (Fraud Caught %)")
ax.set_ylabel("Precision (Of Flagged, % Are Real Fraud)")
ax.set_title("Precision-Recall Curves — Focus on Fraud Class", color="#e0e0e0", fontsize=13)
ax.legend(fontsize=10, framealpha=0.4)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig("act2_pr_curves.png", bbox_inches="tight", dpi=150)
plt.show()
print("[SAVED] act2_pr_curves.png")

# ── Chart 7 of 12: CV Score Distribution ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
positions = [1, 2, 3]
bp = ax.boxplot(
    [cv_scores_dt, cv_scores_rf, cv_scores_xgb],
    positions    = positions,
    patch_artist = True,
    widths       = 0.5,
    medianprops  = dict(color="white", linewidth=2)
)
colors_bp = [COLOR_DT, COLOR_RF, COLOR_XGB]
for patch, color in zip(bp["boxes"], colors_bp):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
for whisker, cap in zip(bp["whiskers"], bp["caps"]):
    whisker.set_color("#aaa")
    cap.set_color("#aaa")
ax.set_xticks(positions)
ax.set_xticklabels(["Decision Tree", "Random Forest", "XGBoost"], fontsize=11)
ax.set_ylabel("ROC-AUC (5-fold CV)")
ax.set_title("CV Score Distribution — Variance Comparison", color="#e0e0e0", fontsize=13)
ax.set_ylim([0.5, 1.0])
for pos, scores in zip(positions, [cv_scores_dt, cv_scores_rf, cv_scores_xgb]):
    ax.text(pos, scores.mean() + 0.005, f"μ={scores.mean():.3f}",
            ha="center", fontsize=10, color="#e0e0e0", fontweight="bold")
plt.tight_layout()
plt.savefig("act2_cv_scores.png", bbox_inches="tight", dpi=150)
plt.show()
print("[SAVED] act2_cv_scores.png")

# ── Chart 8 of 12: Confusion Matrix ───────────────────────────────────────────
best_name = max(
    [("Decision Tree", auc_dt, y_pred_dt),
     ("Random Forest", auc_rf, y_pred_rf),
     ("XGBoost",       auc_xgb, y_pred_xgb)],
    key=lambda x: x[1]
)
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_test, best_name[2])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"],
            ax=ax, linewidths=0.5, linecolor="#333",
            annot_kws={"size": 16, "weight": "bold"})
ax.set_title(f"Confusion Matrix — {best_name[0]} (Best Model)", color="#e0e0e0", fontsize=13)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
tn, fp, fn, tp = cm.ravel()
ax.text(0.02, 0.02,
        f"TN={tn}  FP={fp}\nFN={fn}  TP={tp}",
        transform=ax.transAxes, fontsize=9, color="#e0e0e0",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#2a2a4a", edgecolor="#555"))
plt.tight_layout()
plt.savefig("act2_confusion_matrix.png", bbox_inches="tight", dpi=150)
plt.show()
print("[SAVED] act2_confusion_matrix.png")


## Cell 11 — Act 3: Business Verdict — Cost Matrix & Threshold Optimisation

**A lender doesn't care about accuracy — they care about money lost.**

| Error type | What happens | Cost |
|------------|-------------|------|
| **False Negative (FN)** | Model says *approve* → borrower defaults | Lose full loan principal (~$18,500) |
| **False Positive (FP)** | Model says *reject* → borrower was legit | Lose 1-year interest income (~$1,200) |

**Cost ratio FN:FP ≈ 15:1** — meaning we should reject 15 good applications to prevent 1 default.

**Threshold tuning:** Lower the decision threshold → higher recall (catch more fraud) → lower precision (more false alarms).  
We find the threshold that **minimises total business cost**.


In [ ]:
# Business cost parameters
avg_loan_amount = df_clean["LOAN"].median()     # FN cost = full principal lost
annual_interest = avg_loan_amount * 0.065       # FP cost = ~6.5% APR missed

print(f"  Avg loan amount  : ${avg_loan_amount:,.0f}")
print(f"  FN cost (default): ${avg_loan_amount:,.0f}  (lose full principal)")
print(f"  FP cost (missed) : ${annual_interest:,.0f}   (lose 1-yr interest)")
print(f"  Cost ratio FN:FP : {avg_loan_amount/annual_interest:.1f}x")

# Sweep thresholds 0.05 → 0.95
proba_best  = y_proba_xgb
thresholds  = np.arange(0.05, 0.95, 0.01)
total_costs = []
precisions  = []
recalls     = []
f1_scores   = []

for t in thresholds:
    y_pred_t = (proba_best >= t).astype(int)
    cm_t     = confusion_matrix(y_test, y_pred_t)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    cost = (fn_t * avg_loan_amount) + (fp_t * annual_interest)
    total_costs.append(cost)
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_t, zero_division=0))
    f1_scores.append(f1_score(y_test, y_pred_t, zero_division=0))

# Optimal threshold
opt_idx       = np.argmin(total_costs)
opt_threshold = thresholds[opt_idx]
opt_cost      = total_costs[opt_idx]

default_idx  = np.argmin(np.abs(thresholds - 0.5))
default_cost = total_costs[default_idx]
savings      = default_cost - opt_cost

print(f"\n  Default threshold (0.50) cost : ${default_cost:,.0f}")
print(f"  Optimal threshold ({opt_threshold:.2f}) cost  : ${opt_cost:,.0f}")
print(f"  Estimated savings             : ${savings:,.0f} per test period")

y_pred_opt = (proba_best >= opt_threshold).astype(int)
print(f"\n  Classification Report at threshold {opt_threshold:.2f}:")
print(classification_report(y_test, y_pred_opt, target_names=["Legit", "Fraud/Default"]))


## Cell 12 — Act 3: Feature Importance (RF vs XGBoost)

Feature importance tells the lender **which variables to monitor** in their application form.

- **RF importance** = mean decrease in impurity (MDI)
- **XGBoost importance** = gain (total loss improvement from splits on that feature)

Where **both models agree** → that feature is robust signal, not noise.


In [ ]:
rf_importance  = pd.Series(rf.feature_importances_,  index=FEATURE_COLS).sort_values(ascending=True)
xgb_importance = pd.Series(xgb.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

print("  Top 5 features — Random Forest:")
print(rf_importance.tail(5).round(4).to_string())
print("\n  Top 5 features — XGBoost:")
print(xgb_importance.tail(5).round(4).to_string())


## Cell 13 — Act 3: Full Business Verdict Dashboard

In [ ]:
# ── Chart 9 of 12: Threshold vs Total Cost ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(thresholds, [c / 1000 for c in total_costs], color=COLOR_XGB, lw=2)
ax.axvline(opt_threshold, color=COLOR_FRAUD, lw=2, ls="--",
           label=f"Optimal: {opt_threshold:.2f}")
ax.axvline(0.5, color="#aaa", lw=1.5, ls=":",
           label="Default: 0.50")
ax.scatter([opt_threshold], [opt_cost / 1000], color=COLOR_FRAUD, s=120, zorder=5)
ax.set_xlabel("Decision Threshold")
ax.set_ylabel("Total Business Cost ($K)")
ax.set_title("Threshold Optimisation — Minimise Loan Losses", color="#e0e0e0", fontsize=13)
ax.legend(fontsize=10, framealpha=0.4)
ax.text(opt_threshold + 0.02, (opt_cost / 1000) * 1.05,
        f"Saves ${savings:,.0f}\nvs default",
        color=COLOR_LEGIT, fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig("act3_threshold_cost.png", bbox_inches="tight", dpi=150)
plt.show()
print("[SAVED] act3_threshold_cost.png")

# ── Chart 10 of 12: Precision-Recall Tradeoff by Threshold ────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(thresholds, recalls,    color=COLOR_FRAUD, lw=2, label="Recall (fraud caught)")
ax.plot(thresholds, precisions, color=COLOR_RF,    lw=2, label="Precision (flag accuracy)")
ax.plot(thresholds, f1_scores,  color=COLOR_XGB,   lw=2, label="F1 Score", ls="--")
ax.axvline(opt_threshold, color="#e0e0e0", lw=1.5, ls=":",
           label=f"Optimal threshold: {opt_threshold:.2f}")
ax.set_xlabel("Decision Threshold")
ax.set_ylabel("Score")
ax.set_title("Precision vs Recall Trade-off by Threshold", color="#e0e0e0", fontsize=13)
ax.legend(fontsize=10, framealpha=0.4)
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.savefig("act3_precision_recall_tradeoff.png", bbox_inches="tight", dpi=150)
plt.show()
print("[SAVED] act3_precision_recall_tradeoff.png")

# ── Chart 11 of 12: Feature Importance RF vs XGBoost ──────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
feat_df = pd.DataFrame({
    "Random Forest": rf_importance,
    "XGBoost":       xgb_importance
}).sort_values("Random Forest", ascending=True)
y_pos      = np.arange(len(feat_df))
bar_height = 0.35
ax.barh(y_pos - bar_height/2, feat_df["Random Forest"], height=bar_height,
        color=COLOR_RF,  alpha=0.85, label="Random Forest (MDI)")
ax.barh(y_pos + bar_height/2, feat_df["XGBoost"],       height=bar_height,
        color=COLOR_XGB, alpha=0.85, label="XGBoost (Gain)")
ax.set_yticks(y_pos)
ax.set_yticklabels(feat_df.index, fontsize=10)
ax.set_xlabel("Feature Importance Score")
ax.set_title("Feature Importance — Random Forest vs XGBoost\n(Where both agree = robust signal)", color="#e0e0e0", fontsize=13)
ax.legend(fontsize=10, framealpha=0.4)
plt.tight_layout()
plt.savefig("act3_feature_importance.png", bbox_inches="tight", dpi=150)
plt.show()
print("[SAVED] act3_feature_importance.png")

# ── Chart 12 of 12: Final Model Scorecard ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
ax.axis("off")
scorecard_data = {
    "Metric":            ["ROC-AUC", "Avg Precision", "CV Mean AUC", "CV Std", "FN Cost ($)", "FP Cost ($)"],
    "Decision Tree":     [
        f"{auc_dt:.4f}", f"{ap_dt:.4f}",
        f"{cv_scores_dt.mean():.4f}", f"{cv_scores_dt.std():.4f}",
        f"${confusion_matrix(y_test,y_pred_dt).ravel()[2]*avg_loan_amount:,.0f}",
        f"${confusion_matrix(y_test,y_pred_dt).ravel()[1]*annual_interest:,.0f}",
    ],
    "Random Forest":     [
        f"{auc_rf:.4f}", f"{ap_rf:.4f}",
        f"{cv_scores_rf.mean():.4f}", f"{cv_scores_rf.std():.4f}",
        f"${confusion_matrix(y_test,y_pred_rf).ravel()[2]*avg_loan_amount:,.0f}",
        f"${confusion_matrix(y_test,y_pred_rf).ravel()[1]*annual_interest:,.0f}",
    ],
    "XGBoost (Optimal)": [
        f"{auc_xgb:.4f}", f"{ap_xgb:.4f}",
        f"{cv_scores_xgb.mean():.4f}", f"{cv_scores_xgb.std():.4f}",
        f"${confusion_matrix(y_test,y_pred_opt).ravel()[2]*avg_loan_amount:,.0f}",
        f"${confusion_matrix(y_test,y_pred_opt).ravel()[1]*annual_interest:,.0f}",
    ],
}
sc_df = pd.DataFrame(scorecard_data).set_index("Metric")
tbl = ax.table(
    cellText  = sc_df.values,
    colLabels = sc_df.columns,
    rowLabels = sc_df.index,
    cellLoc   = "center",
    loc       = "center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.3, 2.5)
for (row, col), cell in tbl.get_celld().items():
    cell.set_facecolor("#1a1a2e")
    cell.set_edgecolor("#444")
    cell.set_text_props(color="#e0e0e0")
    if row == 0:
        cell.set_facecolor("#2a2a4a")
        cell.set_text_props(color="#ffca28", fontweight="bold")
    if col == -1:
        cell.set_facecolor("#2a2a4a")
        cell.set_text_props(color="#aaa")
    if col == 1 and row > 0:
        cell.set_facecolor("#1a1a3e")
        cell.set_text_props(color=COLOR_RF)
ax.set_title("Final Model Scorecard — Business Metrics", color="#e0e0e0",
             pad=20, fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("act3_model_scorecard.png", bbox_inches="tight", dpi=150)
plt.show()
print("[SAVED] act3_model_scorecard.png")


## Cell 14 — Pipeline Summary

In [ ]:
winner = max(
    [("Decision Tree", auc_dt), ("Random Forest", auc_rf), ("XGBoost", auc_xgb)],
    key=lambda x: x[1]
)

print("=" * 60)
print("  ✓ LOANSHIELD PIPELINE COMPLETE")
print("=" * 60)
print(f""")
print(f"""  PROJECT   : LoanShield — Lending Fraud & Default Detection Engine
  DATASET   : HMEQ (Home Equity Loan Default) — {len(df_clean):,} rows
  TARGET    : BAD (1=default/fraud, 0=legitimate) | Fraud rate: {y.mean()*100:.1f}%

  MODELS TRAINED:
    1. Decision Tree  (Baseline)     — AUC: {auc_dt:.4f}
    2. Random Forest  (Bagging)      — AUC: {auc_rf:.4f}  ↑ +{auc_rf-auc_dt:.4f} vs baseline
    3. XGBoost        (Boosting)     — AUC: {auc_xgb:.4f}  ↑ +{auc_xgb-auc_dt:.4f} vs baseline

  WINNER    : {winner[0]}  (ROC-AUC: {winner[1]:.4f})

  BUSINESS IMPACT:
    Default threshold  (0.50): Total cost estimate — ${default_cost:,.0f}
    Optimal threshold  ({opt_threshold:.2f}): Total cost estimate — ${opt_cost:,.0f}
    Estimated savings         : ${savings:,.0f} per lending cycle

  TOP FRAUD SIGNALS (both models agree):
    1. DEBTINC  — Debt-to-income ratio
    2. DEROG    — Derogatory credit reports
    3. DELINQ   — Past delinquencies
    4. CLAGE    — Age of credit history
    5. VALUE    — Property value (collateral)

  CONCEPTS DEMONSTRATED:
    ✓ Class imbalance handling (class_weight + scale_pos_weight)
    ✓ Decision Tree — high variance baseline
    ✓ Random Forest — bagging, bootstrap sampling, feature randomness
    ✓ XGBoost       — boosting, sequential error correction, regularisation
    ✓ Stratified K-Fold cross-validation
    ✓ ROC-AUC vs Precision-Recall AUC
    ✓ Business cost matrix — FN vs FP financial impact
    ✓ Threshold optimisation — minimise total loan losses
    ✓ Feature importance comparison across ensemble families
""")


## Cell 15 — Phase 1: Save Model Artifacts for Deployment

We save five artifacts to disk so the FastAPI backend can load them
without retraining:

| File | What it stores |
|------|---------------|
| `random_forest_model.pkl` | Trained Random Forest (winner, AUC 0.9810) |
| `label_encoder_reason.pkl` | Fitted LabelEncoder for REASON column |
| `label_encoder_job.pkl` | Fitted LabelEncoder for JOB column |
| `inference_medians.pkl` | Per-column overall medians for inference-time imputation |
| `model_metadata.pkl` | Threshold, cost params, feature list |

> **Why inference_medians?**  
> At prediction time we receive a single loan application — we can't
> recompute class-conditional medians from one row. We freeze the
> overall training medians so the API imputes consistently with how
> the model was trained.


In [ ]:
import joblib
from pathlib import Path

# Create models/ folder to store all artifacts
Path("models").mkdir(exist_ok=True)

# ── 1. Trained model (Random Forest — winner AUC 0.9810) ─────────────────────
joblib.dump(rf, "models/random_forest_model.pkl")
print("✓ Saved  models/random_forest_model.pkl")

# ── 2. Label encoders (must use the same fitted encoders at inference) ────────
joblib.dump(le_reason, "models/label_encoder_reason.pkl")
joblib.dump(le_job,    "models/label_encoder_job.pkl")
print("✓ Saved  models/label_encoder_reason.pkl")
print("✓ Saved  models/label_encoder_job.pkl")

# ── 3. Inference medians ──────────────────────────────────────────────────────
# Overall median per numeric feature computed from the full cleaned dataset.
# Used to fill any missing fields in a single incoming loan application.
NUMERIC_FEATURES = ["LOAN", "MORTDUE", "VALUE", "YOJ", "DEROG",
                    "DELINQ", "CLAGE", "NINQ", "CLNO", "DEBTINC"]

inference_medians = {
    col: float(df_clean[col].median())
    for col in NUMERIC_FEATURES
}

joblib.dump(inference_medians, "models/inference_medians.pkl")
print("✓ Saved  models/inference_medians.pkl")
print(f"  Medians stored for: {list(inference_medians.keys())}")

# ── 4. Model metadata ─────────────────────────────────────────────────────────
# Everything the API needs to serve predictions without re-running the notebook.
model_metadata = {
    "feature_cols":     FEATURE_COLS,
    "opt_threshold":    float(opt_threshold),
    "avg_loan_amount":  float(avg_loan_amount),
    "annual_interest":  float(annual_interest),
    "fraud_rate":       float(y.mean()),
    "model_auc":        0.9810,
    "reason_classes":   list(le_reason.classes_),
    "job_classes":      list(le_job.classes_),
}

joblib.dump(model_metadata, "models/model_metadata.pkl")
print("✓ Saved  models/model_metadata.pkl")

# ── Verification ──────────────────────────────────────────────────────────────
print("\n── Artifact Verification ──────────────────────────────────────────")
rf_check   = joblib.load("models/random_forest_model.pkl")
meta_check = joblib.load("models/model_metadata.pkl")

test_prob  = rf_check.predict_proba(X_test[:1])[0][1]
print(f"  Model reload check  : predict_proba on 1 sample → {test_prob:.4f}  ✓")
print(f"  Optimal threshold   : {meta_check['opt_threshold']}")
print(f"  Avg loan amount     : ${meta_check['avg_loan_amount']:,.0f}")
print(f"  Features saved      : {meta_check['feature_cols']}")
print(f"  REASON classes      : {meta_check['reason_classes']}")
print(f"  JOB    classes      : {meta_check['job_classes']}")
print("\n✓ All artifacts saved and verified. Ready for Phase 2 — FastAPI backend.")
